# MNIST

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
batch_size = 128
lr = 1e-3
epochs = 10

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.MNIST(root = "./datasets", train = True, download = True, transform = transform)
test_dataset = datasets.MNIST(root = "./datasets", train = False, download = True, transform = transform)

train_loader = DataLoader(dataset = train_dataset, batch_size = batch_size, shuffle = True)
test_loader = DataLoader(dataset = test_dataset, batch_size = batch_size, shuffle = False)

In [ ]:
x, t = next(iter(train_loader))
x = x[0].numpy().transpose(1, 2, 0)
t = t[0].numpy()

print(f"Class = {t}")
plt.imshow(x, cmap = "gray")
plt.show()

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(1 * 28 * 28, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = F.relu(x)
        y = self.fc2(x)

        return y

In [ ]:
model = MLP()
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = lr)

In [ ]:
history = {"loss" : [], "accuracy" : [], "test_loss" : [], "test_accuracy" : []}

for epoch in range(epochs):
    model.train()
    train_loss = 0
    train_accuracy = 0

    for x, t in train_loader:
        x, t = x.to(device), t.to(device)

        y = model(x)
        loss = criterion(y, t)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * x.size(0)
        train_accuracy += (y.argmax(1) == t).sum().item()

    train_loss /= len(train_dataset)
    train_accuracy /= len(train_dataset)

    history["loss"].append(train_loss)
    history["accuracy"].append(train_accuracy)

    model.eval()
    test_loss = 0
    test_accuracy = 0

    with torch.no_grad():
        for x, t in test_loader:
            x, t = x.to(device), t.to(device)
            y = model(x)
            loss = criterion(y, t)
            test_loss += loss.item() * x.size(0)
            test_accuracy += (y.argmax(1) == t).sum().item()

    test_loss /= len(test_dataset)
    test_accuracy /= len(test_dataset)

    history["test_loss"].append(test_loss)
    history["test_accuracy"].append(test_accuracy)

    print(f"Epoch = {epoch + 1}, ", end = "")
    print(f"Loss = {train_loss:.5f}, Accuracy = {train_accuracy:.5f}, ", end = "")
    print(f"Test Loss = {test_loss:.5f}, Test Accuracy = {test_accuracy:.5f}")

In [ ]:
plt.plot(range(1, epochs + 1), history["loss"], label = "Train")
plt.plot(range(1, epochs + 1), history["test_loss"], label = "Test")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, linestyle = "--")
plt.legend()
plt.show()

In [ ]:
plt.plot(range(1, epochs + 1), history["accuracy"], label = "Train")
plt.plot(range(1, epochs + 1), history["test_accuracy"], label = "Test")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(True, linestyle = "--")
plt.legend()
plt.show()

In [ ]:
model.eval()

with torch.no_grad():
    test_accuracy = 0

    class_accuracy = [0.0 for i in range(10)]
    class_datas = [0.0 for i in range(10)]

    for x, t in test_loader:
        x, t = x.to(device), t.to(device)
        y = model(x)
        tmp = y.argmax(1) == t
        test_accuracy += tmp.sum().item()

        c = tmp.squeeze()

        for i in range(len(t)):
            label = t[i].item()
            class_accuracy[label] += c[i].item()
            class_datas[label] += 1

    test_accuracy /= len(test_dataset)
    print(f"Test Accuracy = {test_accuracy:.5f}")

    for i in range(10):
        class_accuracy[i] /= class_datas[i]
        print(f"Accuracy of {i} = {class_accuracy[i]:.5f}")

In [ ]:
plt.bar(range(10), class_accuracy)
plt.xlabel("Class")
plt.ylabel("Accuracy")
plt.xticks(range(10))
plt.ylim(0.0, 1.0)
plt.grid(axis = "y", linestyle = "--")
plt.show()